----------------------------
#### tools in LangChain
-----------------------

In [9]:
from langchain_core.tools import tool

The `@tool` decorator in LangChain is used to turn a regular Python function into a LangChain Tool object. 

This allows the function to be used in LangChain workflows, such as being called by agents or invoked directly.

In [2]:
@tool
def multiply(a: int, b: int) -> int:
   """Multiply two numbers."""
   return a * b

In [3]:
multiply.invoke({"a": 2, "b": 3})

6

In [4]:
print(multiply.name) # multiply
print(multiply.description) # Multiply two numbers.
print(multiply.args) 

multiply
Multiply two numbers.
{'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}


Example 

In [5]:
from langchain_core.messages import ToolMessage

In [6]:
# Simulated API call
# Returns a large dictionary, but we don’t need all fields.
def fetch_user_data(user_id):
    # Simulated response data (this could be a large object)
    response_data = {
        "id": user_id,
        "name": "Alice",
        "age": 30,
        "address": "123 Main St",
        "preferences": ["reading", "hiking", "coding"]
    }
    
    return response_data

In [7]:
# Invoking the tool
user_id      = "12345"
api_response = fetch_user_data(user_id)

# Create a ToolMessage with only relevant metadata
tool_message = ToolMessage(
    content      = {"user_id": api_response["id"], "name": api_response["name"]},
    tool_call_id = "fetch_user_data_tool",
    artifact     = {
            "execution_time": "200ms",
            "data_source": "User API",
            "response_length": len(api_response)
    }
)

print(tool_message)

content="{'user_id': '12345', 'name': 'Alice'}" tool_call_id='fetch_user_data_tool' artifact={'execution_time': '200ms', 'data_source': 'User API', 'response_length': 5}


alternatively ...

In [10]:
import json

In [11]:
# Simulated API call
def fetch_user_data(user_id):
    return {
        "id": user_id,
        "name": "Alice",
        "age": 30,
        "address": "123 Main St",
        "preferences": ["reading", "hiking", "coding"]
    }

In [13]:
# Call API
user_id      = "12345"
api_response = fetch_user_data(user_id)

# Extract only relevant fields
filtered_data = {
    "content": {
        "user_id": api_response["id"],
        "name": api_response["name"]
    },
    "artifact": {
        "execution_time": "200ms",
        "data_source": "User API",
        "response_length": len(api_response)
    }
}

# Convert to JSON
message_json = json.dumps(filtered_data, indent=2)
print(message_json)

{
  "content": {
    "user_id": "12345",
    "name": "Alice"
  },
  "artifact": {
    "execution_time": "200ms",
    "data_source": "User API",
    "response_length": 5
  }
}


OR

In [15]:
from jinja2 import Template


In [17]:
template = Template("""
{
  "content": {
    "user_id": "{{ data.id }}",
    "name": "{{ data.name }}"
  },
  "artifact": {
    "execution_time": "200ms",
    "data_source": "User API",
    "response_length": {{ data | length }}
  }
}
""")

# Render template
output = template.render(data=api_response)
print(output)


{
  "content": {
    "user_id": "12345",
    "name": "Alice"
  },
  "artifact": {
    "execution_time": "200ms",
    "data_source": "User API",
    "response_length": 5
  }
}


In [8]:
from langchain.agents import initialize_agent, AgentType